# YOLOv8s P3SDER Colab Experiment

This notebook clones `yolov8s_kuangjing`, installs the local Ultralytics package from source, and launches the `yolov8s_p3sder.yaml` experiment.

For a real mine dataset run, update `DATA_CFG`, `EPOCHS`, and `BATCH` in the config cell.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/content/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .

In [ ]:
from pathlib import Path
import requests
from PIL import Image

assets_dir = Path("/content/yolov8s_kuangjing/ultralytics/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)

    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / "bus.jpg", "https://ultralytics.com/images/bus.jpg")
ensure_valid_image(assets_dir / "zidane.jpg", "https://ultralytics.com/images/zidane.jpg")
print("assets fixed")

In [ ]:
from pathlib import Path

MODEL_CFG = "ultralytics/cfg/models/v8/yolov8s_p3sder.yaml"
DATA_CFG = "ultralytics/cfg/datasets/coco8.yaml"  # replace with your dataset yaml
EPOCHS = 20
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = 0
PROJECT = "runs/colab"
NAME = "yolov8s_p3sder_colab"
SEED = 42

assert Path(MODEL_CFG).exists(), f"Missing model config: {MODEL_CFG}"
assert Path(DATA_CFG).exists(), f"Missing data config: {DATA_CFG}"

print("MODEL_CFG:", MODEL_CFG)
print("DATA_CFG:", DATA_CFG)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_CFG)
results = model.train(
    data=DATA_CFG,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT,
    name=NAME,
    pretrained="yolov8s.pt",
    cache=False,
    amp=True,
    patience=50,
    seed=SEED,
    deterministic=True,
)
results

In [ ]:
from pathlib import Path

run_dir = Path(PROJECT) / NAME
print("Run directory:", run_dir)
print("results.csv exists:", (run_dir / "results.csv").exists())
print("best.pt exists:", (run_dir / "weights" / "best.pt").exists())
print("last.pt exists:", (run_dir / "weights" / "last.pt").exists())